# 01 · Choice 数据拉取（EmQuantAPI）

本 notebook 负责策略所需**全部原始数据**的拉取与本地缓存（parquet），下游 `src/` 与 `scripts/` 只读缓存、不直接调 API。

**使用方式**
1. 安装 EmQuantAPI（从 Choice 官网下载 Python 版安装包，按说明安装）
2. 在下方「登录」单元填入账号密码（或留空走本机已登录态）
3. 在 `config/macro_indicators.yaml` 中填好 EDB 代码（见 `docs/DATA_FIELDS.md` 待确认清单）
4. 自上而下逐节运行；每节自动**分批 + 缓存 + 失败重试**，重复运行会跳过已缓存数据

**落地文件约定**见 `src/data/loaders.py` 顶部注释。

In [1]:
# ===== 0. 环境与登录 =====
import os, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
RAW = PROJECT_ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

from EmQuantAPI import *

# ---- 填入你的 Choice 账号密码（留空则使用本机 EmQuantAPI 已保存的登录态）----
USERNAME = "pyjijin0536"
PASSWORD = "uf080826"

def login():
    if USERNAME and PASSWORD:
        opts = f"ForceLogin=1,UserName={USERNAME},Password={PASSWORD}"
    else:
        opts = "ForceLogin=1"
    r = c.start(opts)
    if r.ErrorCode != 0:
        raise RuntimeError(f"Choice 登录失败: {r.ErrorCode} {r.ErrorMsg}")
    print("Choice 登录成功")

login()

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:00]:The current version is EmQuantAPI(V2.7.2.0).

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:00]:connect server...

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:00]:user pwd login sucess. This login mode will not update userInfo.

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:01]:user pwd login start success!

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:01]:updating ChoiceToHQ.xml from version 0 to 118

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:02]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-06-12 10:56:02]:DownLoad /Users/shenboheng/Documents/Codex/vendor/EMQuantAPI_Python/python3/libs/mac/bjse_code_conversion.txt success.

Choice 登录成功


In [2]:
# ===== 0.1 通用工具: 重试 / 分批 / parquet 缓存 =====
from src.utils.config import load_yaml

class DataLimitError(RuntimeError):
    """10001029: 单次请求数据点超限(代码数×字段数×天数), 重试无意义, 需缩小请求。
    若连极小的请求也报此错, 则是账号当日/累计流量配额耗尽 —— 等配额恢复后重跑续传。"""

def retry(fn, *args, n_retry=3, sleep=3, **kwargs):
    last = None
    for i in range(n_retry):
        try:
            r = fn(*args, **kwargs)
            ec = getattr(r, "ErrorCode", 0)
            if ec == 10001029:
                raise DataLimitError("10001029 data limit exceeded")
            if ec != 0:
                raise RuntimeError(f"ErrorCode={ec} {getattr(r, 'ErrorMsg', '')}")
            return r
        except DataLimitError:
            raise                      # 不重试: 同样大小的请求只会同样失败
        except Exception as e:
            last = e
            print(f"  retry {i+1}/{n_retry}: {e}")
            time.sleep(sleep * (i + 1))
    raise last

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def save_raw(df: pd.DataFrame, name: str):
    p = RAW / f"{name}.parquet"
    df.to_parquet(p)
    print(f"  saved {p.name}  shape={df.shape}")

def cached(name: str) -> bool:
    return (RAW / f"{name}.parquet").exists()

def csd_to_long(res, value_names):
    """c.csd 结果 -> 长表 [date, code, <fields...>]。res.Data: {code: [[field1序列],[field2序列],...]}"""
    rows = []
    dates = pd.to_datetime(res.Dates)
    for code, mat in res.Data.items():
        for j, d in enumerate(dates):
            rows.append([d, code] + [mat[k][j] for k in range(len(value_names))])
    return pd.DataFrame(rows, columns=["date", "code"] + list(value_names))

STRAT = load_yaml("strategy")
START = STRAT["backtest"]["data_lookback_start"]      # 2014-01-01
END   = STRAT["backtest"]["end_date"]
print(START, "->", END)

2014-01-01 -> 2026-04-17


## 1. 交易日历

⚠️ 若你的 Choice 版本中 `c.tradedates` 参数名不同，请按 Choice「代码生成器」修正。

In [3]:
if not cached("trade_calendar"):
    r = retry(c.tradedates, START, END, "period=1,order=1,market=CNSESH")
    cal = pd.DataFrame({"date": pd.to_datetime(r.Data)})
    save_raw(cal, "trade_calendar")
else:
    print("skip: trade_calendar 已缓存")

skip: trade_calendar 已缓存


## 2. 全A股票池

用板块函数取全部 A 股（含已退市，避免幸存者偏差）。
`001004` 为常用的"全部A股"板块代码 —— **请在 Choice 代码生成器中确认**；如需含退市股请改用对应板块代码并在多个历史日期取并集。

In [4]:
SECTOR_ALL_A = "001004"   # TODO_CONFIRM: 全部A股板块代码

if not cached("stock_universe"):
    codes = set()
    # 在多个历史时点取板块成分并集, 近似覆盖退市股
    for snap in pd.date_range(START, END, freq="YS").strftime("%Y-%m-%d"):
        try:
            r = retry(c.sector, SECTOR_ALL_A, snap)
            codes.update(r.Data[0::2] if isinstance(r.Data, list) else r.Codes)
        except Exception as e:
            print(f"  {snap} 失败: {e}")
    codes = sorted(codes)
    print(f"股票数: {len(codes)}")

    # 基本信息: 上市/退市日期 (字段名 TODO_CONFIRM)
    rows = []
    for batch in chunks(codes, 300):
        r = retry(c.css, ",".join(batch), "NAME,LISTDATE,DELISTDATE", f"EndDate={END}")
        for code, vals in r.Data.items():
            rows.append([code] + vals)
    uni = pd.DataFrame(rows, columns=["code", "name", "list_date", "delist_date"])
    save_raw(uni, "stock_universe")
else:
    print("skip")

skip


## 3. 个股日频行情（预算控制 + 断点续传）

**配额策略**（周配额约1200万、单次运行预算自设）：
- 每个「`CODES_PER_BATCH`只 × 1个自然年」切片拉完**立即**落盘一个分片文件，重跑自动跳过已有分片 → 中断/超额**永不丢进度**
- `QUOTA_BUDGET`：本次运行的数据点预算。代码用交易日历精确预估每个请求的点数，**达到预算前主动停止**，不去撞配额墙（被拒绝的请求也可能计量）
- **按年份从最近往最早拉**：拉满一年是"全部股票×完整年份"，哪怕只完成近几年，下游管线就能先用部分区间调试
- 拉完一部分想先跑下游 → 直接执行下方「3.1 合并分片」单元

每周流程：周一配额重置 → 改 `QUOTA_BUDGET` 为本周可用额度 → 重跑本单元 → 自动从断点续传。

In [5]:
FIELDS_DAILY = "CLOSE,MV,LIQMV,TURN,AMOUNT"      # TODO_CONFIRM: LIQMV(流通市值)字段名仍待确认
OPT_DAILY = "period=1,adjustflag=2,curtype=1,order=1,market=CNSESH,ispandas=0"

CODES_PER_BATCH = 40
QUOTA_BUDGET = 9_500_000     # ← 本次运行允许消耗的数据点数(按你的剩余周配额填, 留~5%安全边际)

PART_DIR = RAW / "stock_daily_parts"; PART_DIR.mkdir(exist_ok=True)
FIELD_NAMES = ["close_adj", "total_mv", "float_mv", "turnover", "amount"]

# 用交易日历精确估算每个年段的交易日数(估算请求点数用)
_cal = pd.to_datetime(pd.read_parquet(RAW / "trade_calendar.parquet")["date"])
def seg_trading_days(s, e):
    n = int(((_cal >= s) & (_cal <= e)).sum())
    return n if n > 0 else 245

def year_segments(start, end, newest_first=True):
    y0, y1 = int(start[:4]), int(end[:4])
    years = range(y1, y0 - 1, -1) if newest_first else range(y0, y1 + 1)
    for y in years:
        yield y, max(start, f"{y}-01-01"), min(end, f"{y}-12-31")

uni = pd.read_parquet(RAW / "stock_universe.parquet")
codes = uni["code"].tolist()
batches = list(chunks(codes, CODES_PER_BATCH))
segs = list(year_segments(START, END))            # 最近年份优先
n_fields = len(FIELD_NAMES)

spent, fetched, skipped, failed = 0, 0, 0, 0
stop_reason = None
for y, seg_s, seg_e in segs:
    if stop_reason:
        break
    days = seg_trading_days(seg_s, seg_e)
    for bi, batch in enumerate(batches):
        part = PART_DIR / f"part_{bi:04d}_{y}.parquet"
        if part.exists():
            skipped += 1
            continue
        est = len(batch) * n_fields * days
        if spent + est > QUOTA_BUDGET:
            stop_reason = (f"达到本次预算: 已耗约{spent:,} + 下一请求约{est:,} "
                           f"> 预算{QUOTA_BUDGET:,}。进度已保留, 配额恢复后重跑续传。")
            break
        try:
            r = retry(c.csd, ",".join(batch), FIELDS_DAILY, seg_s, seg_e, OPT_DAILY)
            csd_to_long(r, FIELD_NAMES).to_parquet(part)
            spent += est
            fetched += 1
        except DataLimitError as e:
            stop_reason = f"配额墙: {e} (实际额度比预算先耗尽, 下周再续)"
            break
        except Exception as e:
            failed += 1
            print(f"  batch{bi}/{y} FAILED: {e} (下次重跑会自动重试该分片)")
        time.sleep(0.3)
    if not stop_reason:
        print(f"{y}年 全部完成 ✓  (本次累计消耗约 {spent:,} 点)")

total_parts = len(batches) * len(segs)
done_parts = len(list(PART_DIR.glob("part_*.parquet")))
print("=" * 60)
if stop_reason:
    print("⏸ ", stop_reason)
print(f"分片进度: {done_parts}/{total_parts} | 本次新拉{fetched}片, 跳过{skipped}片, 失败{failed}片")
print(f"本次消耗约 {spent:,} 点; 剩余约 {(total_parts - done_parts)} 片待拉")
print("可随时运行下方 3.1 单元合并已有分片, 先跑下游管线。")

2026年 全部完成 ✓  (本次累计消耗约 0 点)
2025年 全部完成 ✓  (本次累计消耗约 0 点)
[EmQuantAPI Python] [Em_Info][2026-06-12 09:31:19]:percentflag(for csd/css/cses) update success.



KeyboardInterrupt: 

### 3.1 合并已有分片 → `stock_daily`（可随时执行、可重复执行）

不要求全部分片拉完。合并后打印各年覆盖情况；下游用部分年份即可先调试管线。
每次续传拉完新分片后重跑本单元刷新 `stock_daily`。

In [6]:
parts = sorted(PART_DIR.glob("part_*.parquet"))
if not parts:
    print("尚无分片可合并")
else:
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df = df.dropna(subset=["close_adj"]).drop_duplicates(subset=["date", "code"])
    save_raw(df, "stock_daily")     # 直接覆盖旧合并结果
    cov = df.groupby(df["date"].dt.year)["code"].nunique()
    print("各年覆盖股票数:")
    print(cov.to_string())
    # 估算剩余待拉的配额需求
    have = {p.name for p in parts}
    remain_pts = 0
    for y, seg_s, seg_e in year_segments(START, END):
        days = seg_trading_days(seg_s, seg_e)
        for bi, batch in enumerate(batches):
            if f"part_{bi:04d}_{y}.parquet" not in have:
                remain_pts += len(batch) * len(FIELD_NAMES) * days
    print(f"\n剩余待拉约 {remain_pts:,} 数据点 ≈ {remain_pts/1e7:.1f} 个「千万级周配额」")

  saved stock_daily.parquet  shape=(1881421, 7)
各年覆盖股票数:
date
2024     863
2025    5417
2026    5416

剩余待拉约 71,454,800 数据点 ≈ 7.1 个「千万级周配额」


## 4. 行业分类（月度快照）

报告用 CNE6 的 32 行业；工程近似用**中信一级**（或申万一级）。字段名 TODO_CONFIRM。
取每个月末的行业归属快照即可。

In [7]:
FIELD_INDUSTRY = "CITICCODE2020"   # TODO_CONFIRM: 中信一级行业字段名

if not cached("stock_industry"):
    uni = pd.read_parquet(RAW / "stock_universe.parquet")
    cal = pd.read_parquet(RAW / "trade_calendar.parquet")
    tdays = pd.DatetimeIndex(pd.to_datetime(cal["date"]))
    month_ends = pd.Series(tdays, index=tdays).groupby(tdays.to_period("M")).max()

    rows = []
    for me in month_ends:
        ds = me.strftime("%Y-%m-%d")
        for batch in chunks(uni["code"].tolist(), 500):
            try:
                r = retry(c.css, ",".join(batch), FIELD_INDUSTRY, f"EndDate={ds},TradeDate={ds}")
                for code, vals in r.Data.items():
                    rows.append([me, code, vals[0]])
            except Exception as e:
                print(f"  {ds} batch失败: {e}")
        print(ds, "done")
    save_raw(pd.DataFrame(rows, columns=["date", "code", "industry"]), "stock_industry")
else:
    print("skip")

skip


## 5. 财务数据（描述变量用）

字段全部 TODO_CONFIRM（docs/DATA_FIELDS.md 表B）：资产负债率、营收、净利润、ROA、毛利率、总资产周转率、每股净资产/BPS、EPS(TTM)、近12月股息率等。
按**季度报告期**逐期 c.css 取，注意取数日参数应使用"截至该日已公告"口径以防未来（确认 Choice 是否支持 `ReportDate` + `PublishDate` 双参数；若不支持，需另取公告日期字段自行做 PIT 对齐）。

In [8]:
FIELDS_FIN = "LIBILITYTOASSET,INCOMESTATEMENT_9,INCOMESTATEMENT_60,ROA,GPMARGIN,ASSETTURNRATIOTTM,BPS,EPSTTM,DIVIDENDYIELDY"
# 已按确认清单更新: ATO=ASSETTURNRATIOTTM, 股息率=DIVIDENDYIELDY
# ROA 注意: Choice 该字段为"总资产报酬率"= EBIT*2/(期初+期末总资产), 与Barra的
#   净利润口径ROA略有差异, Phase2对账若有偏差可改用 净利润TTM/总资产 自算

if not cached("stock_financials"):
    uni = pd.read_parquet(RAW / "stock_universe.parquet")
    report_dates = pd.date_range("2012-03-31", END, freq="QE").strftime("%Y-%m-%d")
    rows = []
    for rd in report_dates:
        for batch in chunks(uni["code"].tolist(), 500):
            try:
                r = retry(c.css, ",".join(batch), FIELDS_FIN,
                          f"ReportDate={rd},type=1")   # 参数 TODO_CONFIRM
                for code, vals in r.Data.items():
                    rows.append([rd, code] + list(vals))
            except Exception as e:
                print(f"  {rd} 失败: {e}")
        print(rd, "done")
    cols = ["report_date", "code", "dtoa", "revenue", "net_profit", "roa",
            "gpm", "ato", "bps", "eps_ttm", "div_yield"]
    save_raw(pd.DataFrame(rows, columns=cols), "stock_financials")
else:
    print("skip")

skip


## 6. 宽基指数：日行情 + 月末成分股权重

成分股权重：Choice 报表函数 `c.ctr("INDEXCOMPOSITION", ...)` 或对应专项函数 —— **报表名与参数 TODO_CONFIRM**。

In [9]:
univ_cfg = load_yaml("index_universe")          # 改完 yaml 后这两行必须重新执行
INDEX_CODES = [x["code"] for x in univ_cfg["static_pool"]]

IDX_PART_DIR = RAW / "index_daily_parts"; IDX_PART_DIR.mkdir(exist_ok=True)

# 期望分片清单: 代码 -> 文件路径
expected = {code: IDX_PART_DIR / f"{code.replace('.', '_')}.parquet" for code in INDEX_CODES}
missing = [code for code, p in expected.items() if not p.exists()]
print(f"已有 {len(expected) - len(missing)}/{len(expected)} 个分片, 待补: {missing if missing else '无'}")

bad_codes = []
for code in missing:                  # 只跑缺失的, 已有数据完全不动
    try:
        r = retry(c.csd, code, "CLOSE", START, END, "period=1,order=1,ispandas=0")
        df = csd_to_long(r, ["close"]).dropna(subset=["close"])
        if df.empty:
            print(f"[空数据] {code}: 请在代码生成器确认代码写法")
            bad_codes.append(code)
        else:
            df.to_parquet(expected[code])
            print(f"[OK] {code}  {len(df)}行")
    except DataLimitError:
        print("⏸ csd配额耗尽, 配额恢复后重跑本单元续传")
        break
    except Exception as e:
        print(f"[失败] {code}: {e}")
        bad_codes.append(code)
    time.sleep(0.2)

# 只合并清单内的分片(目录里若有改名遗留的旧文件会被忽略)
if all(p.exists() for p in expected.values()):
    df = pd.concat([pd.read_parquet(p) for p in expected.values()], ignore_index=True)
    save_raw(df, "index_daily")
else:
    still = [c for c, p in expected.items() if not p.exists()]
    print(f"尚缺 {still}, 补齐后会自动合并; 本轮问题代码: {bad_codes}")

已有 18/18 个分片, 待补: 无
  saved index_daily.parquet  shape=(48814, 3)


In [ ]:
if not cached("index_constituents"):
    cal = pd.read_parquet(RAW / "trade_calendar.parquet")
    tdays = pd.DatetimeIndex(pd.to_datetime(cal["date"]))
    month_ends = pd.Series(tdays, index=tdays).groupby(tdays.to_period("M")).max()

    rows = []
    for idx_code in INDEX_CODES:
        for me in month_ends:
            ds = me.strftime("%Y-%m-%d")
            try:
                # TODO_CONFIRM: 报表名/参数名仍未确认。参考你给的签名风格:
                #   c.ctr("IncomeStatementSHSZ", "", "SecuCode=...,ReportDate=...,ReportType=1")
                # 指数成分请在代码生成器"专项报表"里搜"指数成分/成份权重", indicators传""取全列
                r = retry(c.ctr, "INDEXCOMPOSITION", "",
                          f"IndexCode={idx_code},EndDate={ds}")
                data = r.Data  # 通常为 {行号: [secucode, weight]}
                for _, row in (data.items() if isinstance(data, dict) else enumerate(data)):
                    rows.append([me, idx_code, row[0], row[1]])
            except Exception as e:
                print(f"  {idx_code} {ds}: {e}")
        print(idx_code, "done")
    save_raw(pd.DataFrame(rows, columns=["date", "index_code", "code", "weight"]),
             "index_constituents")
else:
    print("skip")

[EmQuantAPI Python] [Em_Error][2026-06-12 09:30:47]:[ctr] [response 103] [error: 10001012, java.lang.Exception: 【10.205.238.131:52302】拒绝请求！原因：【10.205.238.131:52302】拒绝请求！原因：8068047417463540-00000061-10001-无权限[10001]]

[EmQuantAPI Python] [Em_Error][2026-06-12 09:30:47]:[ctr] fail: [10001012] insufficient user access

  retry 1/3: ErrorCode=10001012 insufficient user access


Exception ignored on calling ctypes callback function: <function c.start.<locals>.log at 0x106458680>
Traceback (most recent call last):
  File "/Users/shenboheng/Documents/Codex/vendor/EMQuantAPI_Python/python3/EmQuantAPI.py", line 834, in log
    def log(logMessage):

KeyboardInterrupt: 


[EmQuantAPI Python] [Em_Error][2026-06-12 09:30:54]:[ctr] fail: [10002007] network receive error

  retry 2/3: ErrorCode=10002007 network receive error


## 7. ETF：基本信息 + 日频净值/成交额

- ETF 全集板块代码 TODO_CONFIRM（股票型ETF / 全部ETF板块）
- 关键字段：**跟踪标的指数代码**、上市日期、复权净值、成交额（字段名 TODO_CONFIRM）

In [7]:
SECTOR_ETF = "507003"
FIELDS_ETF_INFO = "NAME,ETFTRACKINDEXCODE,ETFTRACKINDEXNAME"

# 指数池 6位前缀 -> 池内标准代码 (后缀差异通过前缀归一化兼容)
univ_cfg = load_yaml("index_universe")
POOL_BY_PREFIX = {x["code"].split(".")[0]: x["code"] for x in univ_cfg["static_pool"]}

def normalize_tracking(raw_code):
    if raw_code is None or pd.isna(raw_code):
        return None
    prefix = str(raw_code).strip().split(".")[0]
    return POOL_BY_PREFIX.get(prefix)      # 池外指数(行业/主题/债券等)返回None, 自然过滤

if not cached("etf_info"):
    r = retry(c.sector, SECTOR_ETF, END)
    etf_codes = sorted(set(r.Data[0::2] if isinstance(r.Data, list) else r.Codes))
    print("ETF总数:", len(etf_codes))
    rows = []
    for batch in chunks(etf_codes, 200):
        r = retry(c.css, ",".join(batch), FIELDS_ETF_INFO, f"EndDate={END}")
        for code, vals in r.Data.items():
            rows.append([code] + list(vals))
        time.sleep(0.3)
    info = pd.DataFrame(rows, columns=["code", "name", "track_raw", "track_name"])
    info["tracking_index"] = info["track_raw"].map(normalize_tracking)
    info["list_date"] = pd.NaT             # 占位, 单元7.3从首个交易日回填
    save_raw(info, "etf_info")

    mapped = info[info["tracking_index"].notna()]
    print(f"跟踪字段非空: {info['track_raw'].notna().sum()} 只; 映射到指数池: {len(mapped)} 只")
    print(mapped.groupby("tracking_index")["code"].count().to_string())
    # 抽查: 每个池内指数看一只, 核对 track_name 是否对得上
    print("\n抽查(每指数一只, 核对名称):")
    print(mapped.groupby("tracking_index").head(1)
          [["code", "name", "track_raw", "track_name", "tracking_index"]].to_string())
else:
    print("skip: etf_info")

ETF总数: 1566
  saved etf_info.parquet  shape=(1566, 6)
跟踪字段非空: 1539 只; 映射到指数池: 235 只
tracking_index
000010.SH     10
000015.SH      1
000016.SH     12
000300.SH     30
000510.CSI    40
000688.SH     19
000698.SH     14
000852.SH     15
000905.SH     29
000906.SH      4
000922.SH      6
399006.SZ     17
399330.SZ     12
399673.SZ     12
932000.CSI    14

抽查(每指数一只, 核对名称):
           code              name   track_raw track_name tracking_index
140   159205.OF          创业板ETF东财   399006.SZ       创业板指      399006.SZ
146   159211.OF        富国深证100ETF   399330.SZ      深证100      399330.SZ
149   159215.OF       平安中证A500ETF  000510.CSI     中证A500     000510.CSI
170   159238.OF  景顺长城沪深300增强策略ETF   000300.SH      沪深300      000300.SH
221   159298.OF        大成创业板50ETF   399673.SZ      创业板50      399673.SZ
252   159337.OF        中证500ETF基金   000905.SH      中证500      000905.SH
315   159517.OF    银华中证800增强策略ETF   000906.SH      中证800      000906.SH
327   159531.OF       南方中证2000ETF  932000.CSI     中证

In [8]:
ETF_PART_DIR = RAW / "etf_daily_parts"; ETF_PART_DIR.mkdir(exist_ok=True)
ETF_CODES_PER_BATCH = 60                  # 60只×3字段×245天≈4.4万点/请求
FIELDS_ETF_DAILY = "UNITNAVADJ,CLOSE,AMOUNT"          # ⚠️ UNITNAVADJ未验证, 见下方说明
ETF_DAILY_NAMES  = ["nav", "close", "amount"]

def year_segments(start, end, newest_first=True):
    y0, y1 = int(start[:4]), int(end[:4])
    years = range(y1, y0 - 1, -1) if newest_first else range(y0, y1 + 1)
    for y in years:
        yield y, max(start, f"{y}-01-01"), min(end, f"{y}-12-31")

info = pd.read_parquet(RAW / "etf_info.parquet")
pool_etfs = info.loc[info["tracking_index"].notna(), "code"].tolist()
print(f"拉取 {len(pool_etfs)} 只池内ETF (其余{len(info)-len(pool_etfs)}只不拉)")

batches = list(chunks(pool_etfs, ETF_CODES_PER_BATCH))
stop = False
for y, seg_s, seg_e in year_segments(START, END):
    if stop:
        break
    for bi, batch in enumerate(batches):
        part = ETF_PART_DIR / f"part_{bi:04d}_{y}.parquet"
        if part.exists():
            continue
        try:
            r = retry(c.csd, ",".join(batch), FIELDS_ETF_DAILY,
                      seg_s, seg_e, "period=1,order=1,ispandas=0")
            csd_to_long(r, ETF_DAILY_NAMES).to_parquet(part)
        except DataLimitError as e:
            print(f"⏸ 配额耗尽: {e} — 进度已保留, 恢复后重跑续传")
            stop = True
            break
        except Exception as e:
            print(f"  batch{bi}/{y} FAILED: {e} (重跑自动重试)")
        time.sleep(0.3)
    if not stop:
        print(f"{y}年 完成 ✓")

done = len(list(ETF_PART_DIR.glob("part_*.parquet")))
total = len(batches) * len(list(year_segments(START, END)))
print(f"分片进度: {done}/{total}")

拉取 235 只池内ETF (其余1331只不拉)
2026年 完成 ✓
2025年 完成 ✓
2024年 完成 ✓
2023年 完成 ✓
2022年 完成 ✓
[EmQuantAPI Python] [Em_Error][2026-06-12 10:46:19]:can't connect to 61.129.249.40:1818. error = [60]

[EmQuantAPI Python] [Em_Error][2026-06-12 10:46:19]:ZBFW ServerType:3 connect fail, errid:10002003.

[EmQuantAPI Python] [Em_Error][2026-06-12 10:46:19]:[csd] fail: [10002003] network connection timeout

  retry 1/3: ErrorCode=10002003 network connection timeout
2021年 完成 ✓
2020年 完成 ✓
2019年 完成 ✓
2018年 完成 ✓
2017年 完成 ✓
2016年 完成 ✓
2015年 完成 ✓
2014年 完成 ✓
分片进度: 52/52


In [9]:
parts = sorted(ETF_PART_DIR.glob("part_*.parquet"))
if not parts:
    print("尚无分片")
else:
    ed = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    ed = ed.drop_duplicates(subset=["date", "code"])
    if "nav" not in ed.columns:               # 降级到CLOSE方案时补nav列, 下游接口不变
        ed["nav"] = ed["close"]
    save_raw(ed, "etf_daily")
    print(f"etf_daily: {ed['code'].nunique()}只, "
          f"{ed['date'].min().date()} ~ {ed['date'].max().date()}")

    first_trade = ed.dropna(subset=["close"]).groupby("code")["date"].min()
    info = pd.read_parquet(RAW / "etf_info.parquet")
    info["list_date"] = info["code"].map(first_trade)
    save_raw(info, "etf_info")
    print(f"list_date回填: {info['list_date'].notna().sum()}/{len(info)} 只有值"
          "（无值=未拉日频的池外ETF, 不影响）")

  saved etf_daily.parquet  shape=(701945, 5)
etf_daily: 235只, 2014-01-02 ~ 2026-04-17
  saved etf_info.parquet  shape=(1566, 6)
list_date回填: 234/1566 只有值（无值=未拉日频的池外ETF, 不影响）


/var/folders/rn/jxw893090rg08hw65kt7z5cw0000gn/T/ipykernel_31117/1662430632.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ed = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)


In [10]:
if not cached("etf_daily"):
    info = pd.read_parquet(RAW / "etf_info.parquet")
    part_dir = RAW / "etf_daily_parts"; part_dir.mkdir(exist_ok=True)
    ETF_CODES_PER_BATCH = 60          # 3字段, 60×3×245≈4.4万点
    batches = list(chunks(info["code"].tolist(), ETF_CODES_PER_BATCH))
    for bi, batch in enumerate(batches):
        for y, seg_s, seg_e in year_segments(START, END):
            part = part_dir / f"part_{bi:04d}_{y}.parquet"
            if part.exists():
                continue
            try:
                # 复权单位净值 + 收盘价 + 成交额 (字段名 TODO_CONFIRM)
                r = retry(c.csd, ",".join(batch), "UNITNAVADJ,CLOSE,AMOUNT",
                          seg_s, seg_e, "period=1,order=1,ispandas=0")
                csd_to_long(r, ["nav", "close", "amount"]).to_parquet(part)
            except DataLimitError as e:
                print(f"batch{bi}/{y} 超限: {e}")
                raise
            except Exception as e:
                print(f"batch{bi}/{y} FAILED: {e}")
            time.sleep(0.3)
        if bi % 5 == 0:
            print(f"进度: batch {bi}/{len(batches)}")
    parts = sorted(part_dir.glob("part_*.parquet"))
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    save_raw(df, "etf_daily")
else:
    print("skip")

skip


[EmQuantAPI Python] [Em_Error][2026-06-12 10:51:46]:https post fail, nret:35, use http encrypt try again



## 8. 宏观指标（EDB）

EDB 代码已回填 `config/macro_indicators.yaml`（出口/工业/信贷/汇率共 12 个已确认）。

**消费类 4 个指标在 EDB 暂未找到**，处理方式（按优先级）：
1. 换关键词再搜一轮（见下方建议）；找到后回填 yaml 重跑本节即可
2. 仍找不到的 → 用下方「8.1 手动CSV导入」单元从外部数据源补（乘联会官网/灯塔专业版/轻纺城官网均有公开数据）
3. 最终兜底：熵权法按报告规则自动剔除无效指标，消费类哪怕只剩 1~2 个指标也能出分；但 4 个全缺会导致消费类整列缺失，**消费类是报告中区分度最高的一类，建议至少补上 2 个**

**建议搜索关键词**（Choice EDB 浏览器，注意这些多在"行业经济"目录而非"宏观"目录下）：
- 乘用车零售/批发：`乘联会`、`狭义乘用车`、`厂家零售`、`日均销量`（目录：行业经济→汽车）
- 电影票房：`电影票房`、`全国票房`、`观影人次`（目录：行业经济→传媒/文体娱乐，源为国家电影专资办/艺恩）
- 轻纺城成交量：`轻纺城`、`柯桥`（目录：行业经济→纺织服装）

In [11]:
macro_cfg = load_yaml("macro_indicators")
for cat, spec in macro_cfg["categories"].items():
    for ind in spec["indicators"]:
        key = ind["name"]
        if cached(f"macro_{key}"):
            df = pd.read_parquet(RAW / f"macro_{key}.parquet")
            status = f"✓ 已缓存 ({df['date'].min().date()} ~ {df['date'].max().date()}, {len(df)}行)"
        elif not ind.get("enabled", True):
            status = "— 禁用"
        else:
            status = f"待拉, 代码={ind['choice_edb_code']}"
        print(f"{cat:10s} {key:26s} {status}")

consumption 乘用车日均销量_厂家零售_当周            待拉, 代码=EMI01669967
consumption 乘用车日均销量_厂家批发_当周            待拉, 代码=EMI01669969
consumption 电影票房收入                     待拉, 代码=EMI01735960
consumption 中国轻纺城成交量                   — 禁用
export     中国出口集装箱运价指数CCFI_综合指数       待拉, 代码=EMI00107705
export     中国沿海散货运价指数CBFI_综合指数        待拉, 代码=EMI00107703
industrial 三峡水库站_出库流量                 待拉, 代码=EMM01589024
industrial 三峡水库站_入库流量                 待拉, 代码=EMM01589026
industrial 生产资料价格指数                   待拉, 代码=EMM01526663
credit     SHIBOR_1周                  待拉, 代码=E1300076
credit     SHIBOR_1年                  待拉, 代码=E1300082
credit     国债到期收益率_1年                 待拉, 代码=E1002640
credit     国债到期收益率_10年                待拉, 代码=E1002649
credit     DR007                      待拉, 代码=E1300004
credit     中债新综合指数                    待拉, 代码=EMM01590538
fx         美元兑人民币_中间价                 待拉, 代码=E1717692


In [8]:
START_2="2024-01-01"

In [ ]:
macro_cfg = load_yaml("macro_indicators")

for cat, spec in macro_cfg["categories"].items():
    for ind in spec["indicators"]:
        key, edb = ind["name"], ind["choice_edb_code"]
        fname = f"macro_{key}"
        if cached(fname):
            continue
        if not ind.get("enabled", True):
            print(f"[禁用] {key} (yaml中 enabled:false)")
            continue
        if edb in ("TODO_CONFIRM", "MANUAL_CSV"):
            print(f"[跳过] {key}: {'走8.1手动CSV通道' if edb=='MANUAL_CSV' else '请先填EDB代码'}")
            continue
        try:
            r = retry(c.edb, edb, f"StartDate={START_2},EndDate={END},IsLatest=0,ispandas=0")
            dates = pd.to_datetime(r.Dates)
            vals = r.Data[edb][0] if isinstance(r.Data, dict) else r.Data[0]
            df = pd.DataFrame({"date": dates, "value": vals})
            save_raw(df, fname)
        except Exception as e:
            print(f"[失败] {key} ({edb}): {e}")

print("宏观取数结束")

  saved macro_乘用车日均销量_厂家零售_当周.parquet  shape=(116, 2)
  saved macro_乘用车日均销量_厂家批发_当周.parquet  shape=(116, 2)
  saved macro_电影票房收入.parquet  shape=(835, 2)
[禁用] 中国轻纺城成交量 (yaml中 enabled:false)
  saved macro_中国出口集装箱运价指数CCFI_综合指数.parquet  shape=(113, 2)
  saved macro_中国沿海散货运价指数CBFI_综合指数.parquet  shape=(112, 2)
  saved macro_三峡水库站_出库流量.parquet  shape=(762, 2)
  saved macro_三峡水库站_入库流量.parquet  shape=(762, 2)
  saved macro_生产资料价格指数.parquet  shape=(112, 2)
  retry 1/3: ErrorCode=10000009 no data
  retry 2/3: ErrorCode=10000009 no data
  retry 3/3: ErrorCode=10000009 no data
[失败] SHIBOR_1周 (E1300076): ErrorCode=10000009 no data
  retry 1/3: ErrorCode=10000009 no data
  retry 2/3: ErrorCode=10000009 no data
  retry 3/3: ErrorCode=10000009 no data
[失败] SHIBOR_1年 (E1300082): ErrorCode=10000009 no data
  retry 1/3: ErrorCode=10000009 no data
  retry 2/3: ErrorCode=10000009 no data
  retry 3/3: ErrorCode=10000009 no data
[失败] 国债到期收益率_1年 (E1002640): ErrorCode=10000009 no data
  retry 1/3: ErrorCode=100

[EmQuantAPI Python] [Em_Error][2026-06-12 11:26:55]:https post fail, nret:35, use http encrypt try again

[EmQuantAPI Python] [Em_Error][2026-06-12 11:37:19]:https post fail, nret:35, use http encrypt try again

[EmQuantAPI Python] [Em_Error][2026-06-12 11:47:47]:http post fail, nret:28

[EmQuantAPI Python] [Em_Info][2026-06-12 11:47:49]:ServerType:202 switch to net:2.

[EmQuantAPI Python] [Em_Error][2026-06-12 12:18:58]:http post fail, nret:28

[EmQuantAPI Python] [Em_Info][2026-06-12 12:19:04]:ServerType:202 switch to net:1.

[EmQuantAPI Python] [Em_Error][2026-06-12 12:53:16]:XTFW heart error.

[EmQuantAPI Python] [Em_Info][2026-06-12 12:53:16]:[heartbeatthread] server disconnect, reconnect now.

[EmQuantAPI Python] [Em_Info][2026-06-12 12:53:16]:[heartbeatthread] server reconnect success.

[EmQuantAPI Python] [Em_Error][2026-06-12 13:00:27]:Connection closed;error=316 

[EmQuantAPI Python] [Em_Error][2026-06-12 13:00:27]:recv failed: 35

[EmQuantAPI Python] [Em_Error][2026-06-12 13

## 9. 收尾

```python
c.stop()
```
全部跑完后回到项目根目录执行 `python scripts/00_check_data.py` 校验数据完整性。

In [ ]:
c.stop()
print("已登出")